[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/benchuangxd/CSC3109-T16-Project/blob/main/notebooks/07_comparison.ipynb)

# Model Comparison

**CSC3109 - Machine Learning | Team 16**

Aggregates the metrics from all 5 trained models (`results/metrics.csv`) and compares them side by side.

In [ ]:
# -- Google Colab Setup -------------------------------------------------------
# Run this cell first when using Google Colab. No effect when running locally.
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    from pathlib import Path

    REPO_URL  = "https://github.com/benchuangxd/CSC3109-T16-Project.git"
    REPO_PATH = Path("/content/CSC3109-T16-Project")

    if not REPO_PATH.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_PATH)], check=True)
    else:
        print(f"Repo already exists at {REPO_PATH}")

    %cd /content/CSC3109-T16-Project
    %pip install -q -r requirements.txt
    print("Colab setup complete.")
else:
    print("Running locally -- skipping Colab setup.")

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

plt.rcParams["figure.dpi"] = 110
sns.set_style("whitegrid")

METRICS_PATH = ROOT / "results" / "metrics.csv"
FIG_DIR      = ROOT / "report" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Reading metrics from {METRICS_PATH}")

## 1. Metrics Table

In [ ]:
df = pd.read_csv(METRICS_PATH)

# Nicer display names + fixed ordering (lightest to heaviest)
NAME_MAP = {
    "custom_cnn":      "Custom CNN",
    "mobilenet_v3":    "MobileNet V3",
    "efficientnet_b0": "EfficientNet-B0",
    "resnet18":        "ResNet-18",
    "vit_b16":         "ViT-B/16",
}
df["display"] = df["model"].map(NAME_MAP).fillna(df["model"])
order = ["custom_cnn", "mobilenet_v3", "efficientnet_b0", "resnet18", "vit_b16"]
df["model"] = pd.Categorical(df["model"], categories=order, ordered=True)
df = df.sort_values("model").reset_index(drop=True)

display(df.style.format({
    "accuracy": "{:.4f}", "precision": "{:.4f}",
    "recall": "{:.4f}", "f1_score": "{:.4f}", "params_M": "{:.2f}",
}).background_gradient(subset=["accuracy", "f1_score"], cmap="Greens"))

## 2. Performance Bar Chart

In [ ]:
metrics = ["accuracy", "precision", "recall", "f1_score"]
x = np.arange(len(df))
w = 0.2
colors = sns.color_palette("Set2", len(metrics))

fig, ax = plt.subplots(figsize=(12, 5))
for i, m in enumerate(metrics):
    ax.bar(x + (i - 1.5) * w, df[m], w, label=m.replace("_", " ").title(), color=colors[i])

ax.set_xticks(x)
ax.set_xticklabels(df["display"], rotation=10)
ax.set_ylim(0.90, 1.005)
ax.set_ylabel("Score")
ax.set_title("Model Performance Comparison (Validation Split)")
ax.legend(loc="lower right", ncol=4)
ax.grid(axis="y", alpha=0.3)

for i, m in enumerate(metrics):
    for xi, v in zip(x + (i - 1.5) * w, df[m]):
        ax.text(xi, v + 0.001, f"{v:.3f}", ha="center", va="bottom", fontsize=6, rotation=90)

plt.tight_layout()
plt.savefig(FIG_DIR / "model_comparison_bars.png", bbox_inches="tight")
plt.show()

## 3. Accuracy vs Model Size

The ideal model sits in the **top-left**: high accuracy with few parameters.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sizes = df["params_M"] * 12 + 60
ax.scatter(df["params_M"], df["accuracy"], s=sizes,
           c=range(len(df)), cmap="viridis", alpha=0.8, edgecolors="black", zorder=3)

for _, row in df.iterrows():
    ax.annotate(row["display"],
                (row["params_M"], row["accuracy"]),
                textcoords="offset points", xytext=(8, 8), fontsize=9)

ax.set_xscale("log")
ax.set_xlabel("Trainable Parameters (Millions, log scale)")
ax.set_ylabel("Validation Accuracy")
ax.set_title("Accuracy vs Model Size  (bubble size = params)")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / "accuracy_vs_params.png", bbox_inches="tight")
plt.show()

## 4. F1 Ranking

In [ ]:
df_sorted = df.sort_values("f1_score", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(df_sorted["display"], df_sorted["f1_score"],
               color=sns.color_palette("crest", len(df_sorted)))
ax.set_xlim(0.90, 1.005)
ax.set_xlabel("F1 Score (macro)")
ax.set_title("Models Ranked by F1 Score")

for bar, v in zip(bars, df_sorted["f1_score"]):
    ax.text(v + 0.0008, bar.get_y() + bar.get_height()/2,
            f"{v:.4f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / "f1_ranking.png", bbox_inches="tight")
plt.show()

## 5. Best Model Selection

In [ ]:
# Best = highest F1, tie broken by fewest parameters
top = df[df["f1_score"] == df["f1_score"].max()]
best = top.loc[top["params_M"].idxmin()]

print("=" * 50)
print("  BEST MODEL (highest F1, fewest params on tie)")
print("=" * 50)
print(f"  Model     : {best['display']}")
print(f"  Accuracy  : {best['accuracy']:.4f}")
print(f"  F1 Score  : {best['f1_score']:.4f}")
print(f"  Params    : {best['params_M']:.2f} M")
print("=" * 50)
print()
print("Recommendation: use this model for the Docker deployment")
print("(best accuracy-to-size trade-off).")